In [1]:
import json
import re
import pandas as pd
from datasets import Dataset
from itertools import chain
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

C:\Users\ma907\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Vamos a annadir algunas funciones de preprocesamiento para el texto que corresponde con el etiquetado
def delete_emojis(text):
    patron_emojis = re.compile(pattern="["
                                      u"\U0001F600-\U0001F64F"  
                                      u"\U0001F300-\U0001F5FF"  
                                      u"\U0001F680-\U0001F6FF"  
                                      u"\U0001F700-\U0001F77F"  
                                      u"\U0001F780-\U0001F7FF"  
                                      u"\U0001F800-\U0001F8FF"  
                                      u"\U0001F900-\U0001F9FF"  
                                      u"\U0001FA00-\U0001FAFF" 
                                      u"\U00002702-\U000027B0"  
                                      u"\U00002702-\U000027B0"
                                      u"\U000024C2-\U0001F251"
                                      "]+", flags=re.UNICODE)
    return patron_emojis.sub(r'', text)

def delete_commands(texto):
    cleaned_text = re.sub(r'/\S+', '', texto)
    cleaned_text = re.sub(r'@\S+', '', cleaned_text)
    cleaned_text = re.sub(r'#', '', cleaned_text)
    cleaned_text = re.sub(r'http[s]?://\S+', '', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

def normalizar_texto(texto):
    return texto.lower()

In [3]:
with open("barrios_calles_habana.json", "r", encoding="utf-8") as f:
    locations = json.load(f)

all_locations = []
for municipality, neighborhood in locations.items():
    for barrio, calles in neighborhood.items():
        all_locations.append(municipality.lower())  
        all_locations.append(barrio.lower()) 
        all_locations.extend([calle.lower() for calle in calles]) 

def etiquetar_loc(text, locations):
    text = str(text)
    entities = []
    seen = set()  
    for loc in locations:
        for match in re.finditer(r'\b' + re.escape(loc) + r'\b', text):
            if (match.start(), match.end()) not in seen:
                entities.append((match.start(), match.end(), "LOC"))
                seen.add((match.start(), match.end())) 
    return text, {"entities": entities}

def etiquetar_df_column(df, column_name, locations):
    result = []
    for text in df[column_name]:
        _, annotations = etiquetar_loc(text, locations)
        result.append(annotations)
    df[column_name + "_labels"] = result
    return df


In [4]:
data = pd.read_csv("./results---From---2023-10-31--22-08-07---To---2024-04-07--16-02-28.csv")
data = data[['message']].dropna()
data = data.drop_duplicates()
data['message'] = data['message'].apply(delete_emojis)
data['message'] = data['message'].apply(delete_commands) 
data['message'] = data['message'].apply(normalizar_texto)
df = etiquetar_df_column(data, "message", all_locations)
print(df)
                                                                                                    

                                                message  \
0                     renta x días de apto en el vedado   
1     no tienes permisos para ejecutar este comando ...   
2                                                         
8     casa en venta en la zona sur cerca de las fábr...   
9     busco renta por tiempo indefinido para una par...   
...                                                 ...   
4988  busco alquiler en el vedado límite 150 verde s...   
4992     busco alquiler por tiempo indefinido, 58316712   
4993   busco alquiler en la lisa o lo más cerca posible   
4994                          busco alquiler en la lisa   
4996  busco alquiler en playa, marianao, lisa hasta ...   

                                         message_labels  
0      {'entities': [(24, 33, 'geo'), (27, 33, 'geo')]}  
1                                      {'entities': []}  
2                                      {'entities': []}  
8     {'entities': [(25, 28, 'geo'), (120, 123, 'geo...  
9

In [5]:
filtered_df = df[df["message_labels"].notnull() & df["message_labels"].apply(lambda x: len(x["entities"]) > 0)]
filtered_df = filtered_df.drop_duplicates(subset=["message"])

print(filtered_df)

                                                message  \
0                     renta x días de apto en el vedado   
8     casa en venta en la zona sur cerca de las fábr...   
10    busco alquiler en centro habana o la habana vi...   
19    busco urgente alquiler de dos cuartos en marianao   
22    por 10 de octubre preferiblemente de no ser en...   
...                                                 ...   
4986  se alquila apartamento en playa relativamente ...   
4987  hola !!! se renta apartamentico pequeño ( tipo...   
4988  busco alquiler en el vedado límite 150 verde s...   
4993   busco alquiler en la lisa o lo más cerca posible   
4996  busco alquiler en playa, marianao, lisa hasta ...   

                                         message_labels  
0      {'entities': [(24, 33, 'geo'), (27, 33, 'geo')]}  
8     {'entities': [(25, 28, 'geo'), (120, 123, 'geo...  
10    {'entities': [(25, 31, 'geo'), (37, 43, 'geo')...  
19                      {'entities': [(41, 49, 'geo')]}  
2

In [6]:
def convertir_a_conll(df, text_col, labels_col, output_file):
    with open(output_file, "w", encoding="utf-8") as f:
        for text, labels in zip(df[text_col], df[labels_col]):
            entities = labels["entities"]
            label_map = {}
            for start, end, label in entities:
                for i in range(start, end):
                    if i == start:
                        label_map[i] = f"B-{label}"  
                    else:
                        label_map[i] = f"I-{label}" 
            
            tokens = list(re.finditer(r'\S+', text))
            for token in tokens:
                token_start = token.start()
                token_end = token.end()
                token_text = token.group()

                token_label = "O"
                for char_idx in range(token_start, token_end):
                    if char_idx in label_map:
                        token_label = label_map[char_idx]
                        break
                
                f.write(f"{token_text} {token_label}\n")
            
            f.write("\n") 
            
convertir_a_conll(filtered_df, "message", "message_labels", "dataset.conll")


In [7]:
def read_conll(file_path):
    sentences = []
    sentence = []
    labels = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                token, label = line.strip().split()
                sentence.append(token)
                labels.append(label)
            else:
                if sentence:
                    sentences.append({"tokens": sentence, "ner_tags": labels})
                    sentence = []
                    labels = []

    return Dataset.from_list(sentences)

dataset = read_conll("dataset.conll")
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset['train']
test_dataset = dataset['test']


In [18]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=3)


C:\Users\ma907\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ma907\.cache\huggingface\hub\models--dccuchile--bert-base-spanish-wwm-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ImportError: 
TFAutoModelForTokenClassification requires the TensorFlow library but it was not found in your environment.
However, we were able to find a PyTorch installation. PyTorch classes do not begin
with "TF", but are otherwise identically named to our TF classes.
If you want to use PyTorch, please use those classes instead!

If you really do want to use TensorFlow, please follow the instructions on the
installation page https://www.tensorflow.org/install that match your environment.


In [9]:
def clean_dataset(dataset):
    def process_row(row):
        tokens = row["tokens"]
        labels = row["ner_tags"]

        if isinstance(tokens, list) and isinstance(tokens[0], list):
            tokens = tokens[0]
        if isinstance(labels, list) and isinstance(labels[0], list):
            labels = labels[0]

        return {"tokens": tokens, "ner_tags": labels}

    return dataset.map(process_row)

# Limpieza del dataset
train_dataset = clean_dataset(train_dataset)
test_dataset = clean_dataset(test_dataset)


Map: 100%|██████████| 294/294 [00:00<00:00, 6337.79 examples/s]


In [11]:
label_to_id = {
    "O": 0,
    "B-LOC": 1,
    "I-LOC": 2
}

def encode_labels(dataset, label_to_id):
    def process_row(row):
        row["ner_tags"] = [label_to_id[label] for label in row["ner_tags"]]
        return row
    return dataset.map(process_row)

train_dataset = encode_labels(train_dataset, label_to_id)
test_dataset = encode_labels(test_dataset, label_to_id)


Map: 100%|██████████| 294/294 [00:00<00:00, 15471.51 examples/s]


In [12]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",  # Padding uniforme
        max_length=256,  # Ajusta según tu dataset o modelo
        is_split_into_words=True
    )

    labels = examples["ner_tags"]
    new_labels = []
    for i, word_ids in enumerate(tokenized_inputs.word_ids(batch_index=batch_index) for batch_index in range(len(labels))):
        new_labels.append(
            [
                -100 if word_id is None else labels[i][word_id]
                for word_id in word_ids
            ]
        )

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs



train_dataset = train_dataset.map(tokenize_and_align_labels, batched=True)
test_dataset = test_dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/1174 [00:00<?, ? examples/s]

Map: 100%|██████████| 294/294 [00:00<00:00, 3178.66 examples/s]


In [13]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()


C:\Users\ma907\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,No log,0.085157
2,No log,0.068246
3,No log,0.068253


TrainOutput(global_step=441, training_loss=0.10938922942630828, metrics={'train_runtime': 1930.0463, 'train_samples_per_second': 1.825, 'train_steps_per_second': 0.228, 'total_flos': 460147748742144.0, 'train_loss': 0.10938922942630828, 'epoch': 3.0})

In [14]:
results = trainer.evaluate()
predictions, labels, _ = trainer.predict(test_dataset)

In [15]:
flat_predictions = list(chain(*predictions))
flat_labels = list(chain(*labels))

In [16]:
filtered_predictions = [p for p, l in zip(flat_predictions, flat_labels) if l != -100]
filtered_labels = [l for l in flat_labels if l != -100]

filtered_predictions = [np.argmax(p) for p in filtered_predictions]
accuracy = accuracy_score(filtered_labels, filtered_predictions)
precision = precision_score(filtered_labels, filtered_predictions, average='weighted')
recall = recall_score(filtered_labels, filtered_predictions, average='weighted')
f1 = f1_score(filtered_labels, filtered_predictions, average='weighted')

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")



Accuracy: 0.98
Precision: 0.98
Recall: 0.98
F1-Score: 0.98


In [17]:
# Guarda el modelo fine-tuneado en un directorio local
output_dir = "./fine_tuned_model"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Modelo guardado en: {output_dir}")

Modelo guardado en: ./fine_tuned_model
